# 예제 03. RNN과 LSTM의 구조
빅데이터프로그래밍 · 11주차

## 목표
- RNN이 은닉 상태를 다음 시점으로 넘기는 것을 확인한다
- 출력과 은닉 상태의 shape을 읽는다
- LSTM이 왜 필요한지 긴 시퀀스로 확인한다

내부 수식보다 **입력·출력 shape과 은닉 상태의 흐름**에 집중합니다.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)


## 1. RNN 한 단계를 손으로
현재 입력과 이전 은닉 상태를 합쳐 새 은닉 상태를 만듭니다.


In [ ]:
cell = nn.RNNCell(input_size=1, hidden_size=4)

x_seq = torch.tensor([[1.0], [2.0], [3.0]])     # 시점 3개
h = torch.zeros(1, 4)                            # 처음 은닉 상태는 0

for step, x in enumerate(x_seq):
    h = cell(x.unsqueeze(0), h)
    print(f"시점 {step}  입력 {x.item():.1f}  →  은닉 상태 {h.detach().numpy().round(3)}")


은닉 상태가 시점마다 갱신되며 앞의 정보를 실어 나릅니다.


## 2. nn.RNN — 전체 시퀀스를 한 번에


In [ ]:
rnn = nn.RNN(input_size=1, hidden_size=16, batch_first=True)

x = torch.randn(8, 20, 1)          # (batch, 시점, 특성)
out, h_n = rnn(x)

print("입력      :", tuple(x.shape))
print("출력 out  :", tuple(out.shape), "→ 모든 시점의 은닉 상태")
print("마지막 h_n:", tuple(h_n.shape), "→ (층 수, batch, hidden)")


`batch_first=True` 를 꼭 주세요. 없으면 (시점, batch, 특성) 순서가 되어 헷갈립니다.


In [ ]:
# out의 마지막 시점 = h_n
print("같은가:", torch.allclose(out[:, -1, :], h_n[0]))


## 3. 예측에 쓰는 것은 마지막 시점입니다


In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.rnn = nn.RNN(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)         # (batch, 시점, hidden)
        last = out[:, -1, :]         # 마지막 시점만 (batch, hidden)
        return self.fc(last)         # (batch, 1)


m = SimpleRNN()
print("출력:", tuple(m(x).shape))
print("파라미터:", sum(p.numel() for p in m.parameters()))


## 4. LSTM — 은닉 상태가 두 개입니다


In [ ]:
lstm = nn.LSTM(input_size=1, hidden_size=16, batch_first=True)
out, (h_n, c_n) = lstm(x)

print("출력 out:", tuple(out.shape))
print("h_n     :", tuple(h_n.shape), "→ 은닉 상태")
print("c_n     :", tuple(c_n.shape), "→ 셀 상태 (장기 기억)")


LSTM은 셀 상태를 따로 두어 오래된 정보를 더 잘 유지합니다. 사용법은 RNN과 거의 같습니다.


In [ ]:
import pandas as pd

rows = []
for name, layer in [("RNN", nn.RNN(1, 32, batch_first=True)),
                    ("LSTM", nn.LSTM(1, 32, batch_first=True)),
                    ("GRU", nn.GRU(1, 32, batch_first=True))]:
    rows.append({"층": name, "파라미터": sum(p.numel() for p in layer.parameters())})
pd.DataFrame(rows)


LSTM은 RNN의 4배쯤입니다 — 문(gate)이 여러 개이기 때문입니다.


## 5. 긴 시퀀스에서 차이가 납니다
아주 앞의 값을 기억해야 하는 문제를 만들어 비교합니다.


In [ ]:
def make_memory_task(n=2000, length=50):
    """맨 처음 값을 맨 끝에서 맞히는 과제"""
    x = torch.randn(n, length, 1) * 0.1
    first = torch.randn(n, 1)
    x[:, 0, 0] = first.squeeze()          # 첫 시점에만 신호
    return x, first


X, Y = make_memory_task()
print("입력:", tuple(X.shape), "→ 첫 시점 값을 50 시점 뒤에서 맞혀야 합니다")


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

loader = DataLoader(TensorDataset(X, Y), batch_size=64, shuffle=True)
loss_fn = nn.MSELoss()

class Model(nn.Module):
    def __init__(self, kind, hidden=32):
        super().__init__()
        self.rnn = (nn.RNN if kind == "rnn" else nn.LSTM)(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


for kind in ["rnn", "lstm"]:
    torch.manual_seed(42)
    m = Model(kind)
    opt = torch.optim.Adam(m.parameters(), lr=1e-2)
    for epoch in range(15):
        for xb, yb in loader:
            loss = loss_fn(m(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        final = loss_fn(m(X), Y).item()
    print(f"{kind.upper():5s} 최종 손실 {final:.4f}")


길이 50에서 RNN은 첫 값을 거의 잊습니다. LSTM이 더 낮은 손실을 냅니다.


## 6. 층을 여러 개 쌓기


In [ ]:
deep = nn.LSTM(1, 32, num_layers=2, batch_first=True, dropout=0.2)
out, (h, c) = deep(x)
print("out:", tuple(out.shape))
print("h_n:", tuple(h.shape), "→ 층이 2개라 첫 축이 2")


## 7. 자주 만나는 오류 — 3차원이 아니다


In [ ]:
flat = torch.randn(8, 20)          # 특성 축이 없음
try:
    nn.LSTM(1, 16, batch_first=True)(flat)
except RuntimeError as err:
    print("RuntimeError:", err)

print("\n해결:", tuple(nn.LSTM(1, 16, batch_first=True)(flat.unsqueeze(-1))[0].shape))


## 직접 해보기
1. `hidden_size` 를 64로 늘리면 파라미터가 몇 개가 되나요?
2. 시퀀스 길이를 100으로 늘리면 RNN과 LSTM의 차이가 더 벌어지나요?
3. GRU로도 같은 실험을 해 보세요.


In [ ]:
# 여기에 작성하세요
